```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 1: DRL on Flat Terrain

**Goal:** train a policy that makes the Go2 follow velocity commands in any direction on flat ground, then test one design choice of your own in a controlled experiment.

| Section | What you do |
| --- | --- |
| 1.1 | Build the environment, i.e. the MDP the policy learns in |
| 1.2 | Configure the policy network and the PPO algorithm |
| 1.3 | Train a baseline and monitor the training |
| 1.4 | Test the policy and collect evaluation data |
| 1.5 | Run a controlled experiment against your baseline |

This notebook **produces data**: trained policies, training logs and evaluation rollouts, all saved in `logs/`. 
It **requires a GPU for training**, evaluation and visualization can possibly be done on CPU but at a much slower rate.
You analyse the produced data in `logs/` in `02_flat_analysis.ipynb`, which is lightweight and runs entirely on CPU.
Take this into account when dividing work within your group!

**Time budget.** A training run takes roughly 10 to 20 minutes of shared GPU time. 
Check your environment in the viewer (1.1h) before training, and change one thing at a time between runs so you know what caused a difference.

If you are using gnoto, the GPUs might be busy during the practicals and **close to the deadline**.
Make sure to plan your training runs accordingly!

In [2]:
%load_ext autoreload
%autoreload 2

import math
import copy
import lr_project # Before mjlab import

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.rl import RslRlModelCfg, RslRlPpoAlgorithmCfg, RslRlOnPolicyRunnerCfg

from mjlab.managers.action_manager import ActionTermCfg
from mjlab.managers.command_manager import CommandTermCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.observation_manager import ObservationTermCfg, ObservationGroupCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.managers.termination_manager import TerminationTermCfg

from lr_project import mdp, scene, play, train, evaluate, registry

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1.1 Building the environment configuration

In mjlab, an environment is described by a configuration class, the `ManagerBasedRLEnvCfg`.
It is built from several parts, each defining one piece of the Markov Decision Process (MDP) which you will define in the sections below:


| MDP element | Config part | Section |
| --- | --- | --- |
| Task (what the robot should do) | commands | 1.1a |
| Observation $o_t$ | observations | 1.1b |
| Action $a_t$ | actions | 1.1c |
| Reward $r_t$ | rewards | 1.1d |
| End of an episode | terminations | 1.1e |
| Initial state distribution | events | 1.1f |
| Transition dynamics $P$ | MuJoCo physics (provided by `scene`) | 1.1g |

Each `make_*` function you will complete below returns a dictionary `{name: term config}`.
A term is a **function** (`func`) plus its optional **parameters** (`params`).
We provide you with ready-made functions in `src/lr_project/mdp/*.py` (skim through it!), and mjlab provides more in `mjlab.envs.mdp`.
You can also write your own functions in a cell and use it!

For more information on mjlab's syntax and available functions, consult the [official documentation](https://mujocolab.github.io/mjlab/main/index.html).


Every `make_*` function receives a `play` argument.
It is false for training (4096 parallel robots) and true for visualization and evaluation (1 robot, debug rendering of the commands).
Use it if a term should behave differently in the two cases.

**Advice:** it can be a bit daunting to define everything from scratch, where exactly do you start?
A helpful source of inspiration is taking a look at existing literature, for example the articles from the student presentations.
A starting point we recommend is the following paper by Rudin et al., which is a solid baseline for training quadruped locomotion policies using GPUs:

> Rudin, N., Hoeller, D., Reist, P. &amp; Hutter, M.. (2022). Learning to Walk in Minutes Using Massively Parallel Deep Reinforcement Learning. <i>Proceedings`of the 5th Conference on Robot Learning</i>, in <i>Proceedings of Machine Learning Research</i> 164:91-100 Available from https://proceedings.mlr.press/v164/rudin22a.html.

Note that this article also does challenging terrain, so implementing it exactly would be overkill for this policy that only needs to walk on flat ground.

### 1.1a Commands

The command defines the task: 
a desired base velocity $(v_x, v_y, \omega_z)$ in the robot's frame (forward and lateral speed in m/s, yaw rate in rad/s). 
Each robot samples a command uniformly from the ranges below and samples a new one every `resampling_time_range` seconds. 
The policy sees the command as an observation (1.1b) and is rewarded for tracking it (1.1d).

Choose ranges the Go2 can realistically reach. 
Things to think about:
- Why resample the command during an episode instead of once per episode?
- What happens if no robot is ever commanded to stand still (`rel_standing_envs`)?
- `rel_forward_envs` is the fraction of robots that only walk forward. Why could that help or hurt?

**Note:** there is a bug in mjlab's viewer when the command ranges are identical (it crashes), make sure to define them to realistic ranges before running the viewer.

In [3]:
def make_commands(play: bool) -> dict[str, CommandTermCfg]:
    return {
        "joystick": mdp.UniformVelocityCommandCfg(
            resampling_time_range=(10.0, 10.0), # new command every 10 s: about 2 commands per 20 s episode
            rel_standing_envs=0.1, # 10% of robots are commanded to stand still
            rel_forward_envs=0.1, # 10% of robots only walk forward (most common use case)
            ranges=mdp.UniformVelocityCommandCfg.Ranges(
                lin_vel_x=(-1.0, 1.0), # m/s
                lin_vel_y=(-0.5, 0.5), # m/s, lateral walking is harder for a quadruped
                ang_vel_z=(-1.0, 1.0), # rad/s
            ),
            entity_name="robot",
            debug_vis=play, # Do not render command arrows during training
        )
    }

### 1.1b Observations

Observations are what the networks see at every environment step. 
There are two groups:

- **actor**: the input of the policy. 
The policy will run on the robot, so think about what a real Go2 can measure or know before adding the observation.
- **critic**: the input of the value function. 
The critic is only used during training, so it may also receive *privileged* information that is easy to get in simulation but hard to measure on hardware (this is called an asymmetric actor-critic).

Terms are concatenated in the order you list them. 
Available functions are in `src/lr_project/mdp/observations.py`.

Things to think about:
What does the policy need to know at inference time?
Can the real robot make the measurements for the policy?
Can the critic infer the reward from its observations alone?

In [4]:
def make_observations(play: bool) -> dict[str, ObservationGroupCfg]:
    actor = {
        "command": ObservationTermCfg(
            func=mdp.generated_command,
            params={"command_name": "joystick"},
        ),
        # Base state (IMU + state estimator on the real robot)
        "base_lin_vel": ObservationTermCfg(func=mdp.base_lin_vel_b),
        "base_ang_vel": ObservationTermCfg(func=mdp.base_ang_vel_b),
        "projected_gravity": ObservationTermCfg(func=mdp.projected_gravity_b),
        # Joint state (encoders on the real robot), positions relative to the default pose
        "joint_pos": ObservationTermCfg(func=mdp.joint_positions),
        "joint_vel": ObservationTermCfg(func=mdp.joint_velocities),
        # Previous action, so the policy knows what it just commanded
        "last_action": ObservationTermCfg(func=mdp.last_action),
    }

    critic = {
        **actor, # the critic sees everything the actor sees...
        # ...plus privileged foot contact information (not reliably measured on the Go2)
        "foot_contact": ObservationTermCfg(
            func=mdp.foot_contact,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
        "foot_air_time": ObservationTermCfg(
            func=mdp.foot_air_time,
            params={"sensor_name": scene.FEET_CONTACT_SENSOR},
        ),
    }

    return {
        "actor": ObservationGroupCfg(terms=actor),
        "critic": ObservationGroupCfg(terms=critic),
    }

### 1.1c Actions

The policy outputs 12 numbers, one per joint. 
With `JointPositionActionCfg`, each output is turned into a joint position target:

$$q^{target} = q^{default} + \text{scale} \cdot a_t \quad (\text{with } \texttt{use\_default\_offset=True})$$

A PD controller in the robot model then computes the motor torques that move the joint towards this target. 
The policy runs at 50 Hz, the PD controller and physics at 200 Hz.

Things to think about: Why predict offsets from the standing pose instead of absolute angles? What happens if `scale` is too small or too large?

While this is a common action space for quadrupeds, there are many other ways to translate policy actions into motor commands, and these can have a huge effect on your policy's performance!
For instance, one can use the policy outputs to modulate a CPG, as done previously in our lab:

> Bellegarda, G., & Ijspeert, A. (2022). CPG-RL: Learning central pattern generators for quadruped locomotion. IEEE Robotics and Automation Letters, 7(4), 12547-12554.

If interested in the topic, check out this paper for more action spaces and their effects on learning:

> Peng, X. B., & Van De Panne, M. (2017, July). Learning locomotion skills using deeprl: Does the choice of action space matter?. In Proceedings of the ACM SIGGRAPH/Eurographics Symposium on Computer Animation (pp. 1-13).

In [5]:
def make_actions(play: bool) -> dict[str, ActionTermCfg]:
    return {
        "joint_targets": mdp.JointPositionActionCfg(
            entity_name="robot",
            actuator_names=(".*"), # all 12 joints
            use_default_offset=True, # action 0 = default standing pose
            scale=0.25, # rad of joint motion per unit of action
        ),
    }

### 1.1d Rewards

At every step the robot receives $r_t = \sum_i w_i \, f_i(s_t, a_t) \, \Delta t$: 
each term $f_i$ is multiplied by its weight $w_i$ (positive = reward, negative = penalty) and by the policy time step $\Delta t$.
Available functions are in `src/lr_project/mdp/rewards.py`:

- **Task**: `track_linear_velocity`, `track_angular_velocity` reward $\exp(-\text{error}^2 / \text{std}^2)$, so `std` sets how tolerant the tracking is (plot this function for better intuition!)
- **Penalties** (regularization): body velocities, joint velocities, accelerations, torques and power, action rate, contacts, ...
- **Shaping**: `feet_air_time` encourages stepping over jittering.

Terms that take a `sensor_name` need a sensor defined in the scene (the feet contact sensor is called `scene.FEET_CONTACT_SENSOR`).
Check `src/lr_project/scene.py` for more sensor names.

**How to tune:** 
start with the tracking terms and a few (minimal) penalties, then add or change **one term at a time**. 
Watch each term's curve (`Episode_Reward/<name>`) in TensorBoard (see 1.3a). 
If the robot finds an unexpected way to collect reward (e.g. vibrating in place), ask which term allows it. 

Keep a short log of what you tried!
You will explain your choices in notebook 02 and we will ask about them during the oral exam.

In [6]:
def make_rewards(play: bool) -> dict[str, RewardTermCfg]:
    return {
        # Task
        "track_linear_velocity": RewardTermCfg(
            func=mdp.track_linear_velocity,
            weight=1.0,
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),
        "track_angular_velocity": RewardTermCfg(
            func=mdp.track_angular_velocity,
            weight=0.5,
            params={"command_name": "joystick", "std": math.sqrt(0.25)},
        ),
        # Base stability
        "lin_vel_z": RewardTermCfg(func=mdp.body_linear_velocity_penalty, weight=-2.0),
        "ang_vel_xy": RewardTermCfg(func=mdp.body_angular_velocity_penalty, weight=-0.05),
        "flat_orientation": RewardTermCfg(func=mdp.flat_orientation_l2, weight=-1.0),
        # Smoothness and efficiency
        "joint_torques": RewardTermCfg(func=mdp.joint_torques_l2, weight=-1.0e-5),
        "joint_acc": RewardTermCfg(func=mdp.joint_acc_l2, weight=-2.5e-7),
        "action_rate": RewardTermCfg(func=mdp.action_rate_l2, weight=-0.01),
        # No thigh, hip or calf hitting the ground
        "leg_contacts": RewardTermCfg(
            func=mdp.contacts_penalty,
            weight=-1.0,
            params={"sensor_name": scene.LEG_CONTACT_SENSOR},
        ),
        # Shaping: reward stepping instead of shuffling, only when asked to move
        "feet_air_time": RewardTermCfg(
            func=mdp.feet_air_time,
            weight=0.25,
            params={
                "sensor_name": scene.FEET_CONTACT_SENSOR,
                "command_name": "joystick",
                "threshold_min": 0.05,
                "threshold_max": 0.5,
            },
        ),
    }

### 1.1e Terminations

An episode ends when any termination term returns `True` for a robot, and that robot is reset (1.1f).
There are two kinds:

- **Time-outs** (`time_out=True`): 
the episode reached its time limit. 
This is not a failure: PPO keeps estimating the value of the last state, as if the robot could continue (why is this necessary?)
- **Failures** (`time_out=False`): 
the robot fell. 
The robot loses all future rewards, which acts as an implicit penalty, and no more samples are wasted on a robot lying on the ground.

Things to think about: 
What should count as a fall? 
What happens if the failure condition is too strict or too loose?

In [7]:
def make_terminations(play: bool) -> dict[str, TerminationTermCfg]:
    return {
        # End of the 20 s episode: not a failure, the robot could keep walking
        "time_out": TerminationTermCfg(
            func=mdp.time_out,
            time_out=True,
        ),
        # Fall: the base (trunk) touches the ground
        "base_contact": TerminationTermCfg(
            func=mdp.illegal_contact,
            params={"sensor_name": scene.BASE_CONTACT_SENSOR},
            time_out=False,
        ),
        # Fall: the base is tilted by more than ~57 degrees
        "bad_orientation": TerminationTermCfg(
            func=mdp.bad_orientation,
            params={"limit_angle": 1.0},
            time_out=False,
        ),
    }

### 1.1f Events

Events are functions that modify the simulation at given moments, set by `mode`:

- `"startup"`: once, when the environment is created;
- `"reset"`: every time a robot starts a new episode;
- `"interval"`: periodically during an episode (e.g. pushes).

Here, reset events define the **initial state distribution**: where and how robots start. 
`pose_range` gives ranges for the base position (m, relative to its spawn point) and orientation (rad). 
`reset_joints_by_offset` can also randomize the initial joint angles.

Things to think about:
Why do the initial poses need to be randomized?
Is it as important on flat terrain than on a challenging terrain scenario?

In [8]:
def make_events(play: bool) -> dict[str, EventTermCfg]:
    return {
        "reset_base": EventTermCfg(
            func=mdp.reset_root_pose_uniform,
            mode="reset", # at the start of every episode
            params={
                "pose_range": {
                    "x": (-0.5, 0.5), # m
                    "y": (-0.5, 0.5), # m
                    "z": (0.0, 0.05), # m, slightly above the default height to avoid ground penetration
                    "roll": (-0.1, 0.1), # rad
                    "pitch": (-0.1, 0.1), # rad
                    "yaw": (-math.pi, math.pi), # rad, any heading
                }
            }
        ),
        # Reset joints to the nominal standing pose, otherwise a robot keeps the
        # (possibly fallen) joint configuration of its previous episode
        "reset_joints": EventTermCfg(
            func=mdp.reset_joints_by_offset,
            mode="reset",
            params={
                "position_range": (0.0, 0.0), # rad
                "velocity_range": (0.0, 0.0), # rad/s
            },
        ),
    }

### 1.1g Assembly

All parts come together here, with the scene (robot, flat terrain, feet contact sensor) and the simulator settings. 
The physics runs at 200 Hz (`timestep_s=0.005`) and the policy acts every 4 physics steps (`decimation=4`), i.e. at 50 Hz. 
An episode lasts 20 s = 1000 policy steps. 
You normally do not need to change this cell.

In [9]:
def make_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    return ManagerBasedRlEnvCfg(
        scene=scene.make_scene_cfg(
            num_envs=1 if play else 4096,
            terrain="flat",
            robot_entity_name="robot",
        ),
        sim=scene.make_sim_cfg(timestep_s=0.005),
        actions=make_actions(play),
        commands=make_commands(play),
        events=make_events(play),
        observations=make_observations(play),
        rewards=make_rewards(play),
        terminations=make_terminations(play),
        decimation=4,
        episode_length_s=20.0,
        seed=seed,
    )

### 1.1h Visualize your environment

Run your environment with a `"random"` or `"zero"` policy before training. 
Check that robots start where you expect, that the command arrows look right, and that a fallen robot is reset. 
Problems found here cost seconds while problems found after training cost a training run!

Stop the viewer with the **interrupt** button (■) before running the next cell.

In [16]:
env_cfg = make_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

Setting seed: 42
Warp 1.17.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla V100-SXM2-32GB" (9 GiB, sm_70, mempool enabled)
   Kernel cache:
     /home/.cache/warp/1.17.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 10.86 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 5.67 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_9914afce 9914afc load on device 'cuda:0' took 10.11 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 5.90 ms  (cached)
Module _friction_dof__locals__kernel_35a89db9 35a89db load on device 'cuda:0' took 6.63 ms  (cached)
Module _limit_slide_hinge__locals__kernel_6395e840 6395e84 load on device 'cuda:0' took 13.09 ms  (cached)
Module _efc_contact_init__locals__kernel_81f4db55 81f4db5 load on device 'cuda:0' took 9.94 ms  (cached)
Module _efc_contact_jac_dense__locals__kernel_

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: https://gnoto.epfl.ch/user-redirect/proxy/8080


(viser) Connection opened (0, 1 total), 262 persistent messages

(viser) Connection closed (0, 0 total)


Ctrl+C received. Shutting down viewer...


(viser) Server stopped

Viewer closed!


/home/gnoto_venvs/lr-project/lib/python3.13/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


## 1.2 Building the training configuration

The policy is trained with **PPO** (Proximal Policy Optimization), an actor-critic algorithm from the `rsl_rl` library. 
Each iteration has two phases:

1. **Collect:** all 4096 robots run the current policy for `num_steps_per_env` steps.
2. **Update:** the actor and critic networks are improved using the collected data.

The values below are a reasonable starting point. 
**Change them only with a reason** (i.e., after having a working baseline), and one at a time.

### 1.2a Actor model

The actor is the policy: a neural network (MLP) mapping the actor observations to the mean of a Gaussian distribution over actions. 
During training, actions are sampled from this distribution to explore: `init_std` is the initial exploration noise, and the standard deviation is learned. 
`obs_normalization` rescales each observation using running statistics (why is this needed?).

In [10]:
def make_actor_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
        distribution_cfg={
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
            "std_type": "scalar",
        },
    )

### 1.2b Critic model

The critic estimates the value $V(s)$: 
the expected discounted sum of future rewards from a state. 
PPO uses it to judge whether an action turned out better or worse than expected (the *advantage*). 
It takes the critic observations as input and is not used once training is finished.

In [11]:
def make_critic_cfg() -> RslRlModelCfg:
    return RslRlModelCfg(
        hidden_dims=(128, 128, 128),
        activation="elu",
        obs_normalization=True,
    )

### 1.2c PPO hyperparameters

| Parameter | Meaning |
| --- | --- |
| `gamma` | Discount factor $\gamma$: how much future rewards count. $\gamma = 0.995$ at 50 Hz looks about 4 s ahead. |
| `lam` | GAE $\lambda$: trade-off between bias and variance of the advantage estimate. |
| `clip_param` | Limits how much the policy can change in one update (stabilizes training). |
| `entropy_coef` | Bonus for keeping the action distribution wide (exploration). |
| `num_learning_epochs`, `num_mini_batches` | How many passes over the collected data, split into how many batches. |
| `learning_rate`, `schedule`, `desired_kl` | Learning rate for fitting the neural networks, `"adaptive"` lowers it if the policy changes more than `desired_kl` per update. |
| `value_loss_coef`, `use_clipped_value_loss` | Weight and clipping of the critic loss. |
| `max_grad_norm` | Gradient clipping, for stability. |


In [12]:
def make_ppo_cfg() -> RslRlPpoAlgorithmCfg:
    return RslRlPpoAlgorithmCfg(
        value_loss_coef=1.0,
        use_clipped_value_loss=True,
        clip_param=0.2,
        entropy_coef=0.01,
        num_learning_epochs=5,
        num_mini_batches=4,
        learning_rate=1.0e-3,
        schedule="adaptive",
        gamma=0.995,
        lam=0.97,
        desired_kl=0.01,
        max_grad_norm=1.0,
    )

### 1.2d Assembly

`make_rl_cfg` combines the actor-criticm models with the PPO hyperparameters, and sets the size of training: 
each iteration collects 4096 robots × 24 steps ≈ 98k transitions, for `max_iterations` iterations. 
A checkpoint is saved every `save_interval` iterations and at the end. 
The `seed` makes a run reproducible: train the same configuration with different seeds to see how much results vary by chance!

**Do not change the `max_iterations` argument!**
For flat terrain locomotion, your policies should comfortably converge within 500 iterations (and keeps training within acceptable time frames for shared GPU usage).
If it doesn't, tune your MDP further.

In [13]:
def make_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        seed=seed,
        num_steps_per_env=24,
        max_iterations=500, # DO NOT INCREASE
        save_interval=500,
        actor=make_actor_cfg(),
        critic=make_critic_cfg(),
        algorithm=make_ppo_cfg(),
    )

## 1.3 Training

In this section you will run and monitor your training using the previously defined MDP and RL configurations.

### 1.3a Monitoring your training with Tensorboard

Start TensorBoard **before** training, then open the printed link in a new tab. 
Refresh the page to update the curves while training runs.
Useful curves to monitor and compare across trainings:

- `Train/mean_reward` and `Train/mean_episode_length` (1000 steps = no robot falls within 20 s);
- `Episode_Reward/<term>`: the contribution of each reward term, to see what the policy is optimizing;
- `Loss/*` and the policy's action standard deviation, to see whether learning is stable and exploration shrinks.

In [14]:
train.tensorboard()

Tensorboard server running! Monitor your training here: https://gnoto.epfl.ch/user-redirect/proxy/36273/


### 1.3b Running your training

Training saves everything in `logs/<experiment>/<date>_<variant>/`. 
Running the cell again starts a **new** run, previous runs are kept.
Make sure to back up your `logs` as producing the data and checkpoints is computationally expensive.

- Use `experiment="flat"` for all runs of this task, and `variant` to name what the run tests (initially your baseline)
- To train the same configuration with another seed, pass `seed=...` to both `make_env_cfg` and `make_rl_cfg` and keep the same variant name.

In [29]:
train.train(
    env_cfg=make_env_cfg(),
    rl_cfg=make_rl_cfg(),
    experiment="flat",
    variant="baseline",
)

[train] run dir: /home/rl_unitree_go2/logs/flat/2026-11-11_18-27-30_baseline
[train] device: cuda:0, envs: 4096, seed: 42
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+


PosixPath('/home/rl_unitree_go2/logs/flat/2026-11-11_18-27-30_baseline')

#### Seed repeats of the frozen baseline

Once the baseline is frozen (no more tuning), it is trained with two more seeds so notebook 02 can compare the difference between configurations with the spread between seeds. Seed 42 is the run of the cell above. Each run takes about 12 minutes.

In [15]:
import yaml

# Seeds used for every configuration compared in notebook 02
SEEDS = [42, 1, 2]


def run_seed(r) -> int:
    """Seed of a run, read from the config saved with it."""
    return yaml.unsafe_load((r.path / "params" / "agent.yaml").read_text())["seed"]


def runs_per_seed(variant: str) -> dict:
    """Latest complete run of `variant` for each seed in SEEDS (seed -> Run)."""
    found = {}
    for r in registry.runs(experiment="flat", variant=variant):  # sorted by creation time
        if any(r.path.glob("model_499.pt")) and run_seed(r) in SEEDS:
            found[run_seed(r)] = r
    return found

In [16]:
for seed in SEEDS:
    if seed in runs_per_seed("baseline"):
        print(f"seed {seed}: already trained, skipped")
        continue
    train.train(
        env_cfg=make_env_cfg(seed=seed),
        rl_cfg=make_rl_cfg(seed=seed),
        experiment="flat",
        variant="baseline",
    )

seed 42: already trained, skipped
[train] run dir: /home/rl_unitree_go2/logs/flat/2026-11-12_10-51-16_baseline
[train] device: cuda:0, envs: 4096, seed: 1
Setting seed: 1
Warp 1.17.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla V100-SXM2-32GB" (9 GiB, sm_70, mempool enabled)
   Kernel cache:
     /home/.cache/warp/1.17.0
Module mujoco_warp._src.smooth b9cc505 load on device 'cuda:0' took 13.70 ms  (cached)
Module _nxn_broadphase__locals__kernel_3a0c7ab9 3a0c7ab load on device 'cuda:0' took 13.44 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_9914afce 9914afc load on device 'cuda:0' took 7.74 ms  (cached)
Module mujoco_warp._src.constraint 9435a53 load on device 'cuda:0' took 6.20 ms  (cached)
Module _friction_dof__locals__kernel_35a89db9 35a89db load on device 'cuda:0' took 8.48 ms  (cached)
Module _limit_slide_hinge__locals__kernel_6395e840 6395e84 load on device 'cuda:0' took 16.35 ms  (cache

### 1.3c Loading your latest training directory

The registry finds runs by experiment and variant, so this cell also works after a kernel restart: 
you never need to retrain to continue and can skip the training cell. 
`runs(...)` returns the runs sorted by creation time, and `[-1]` takes the latest.

In [19]:
baseline_runs = runs_per_seed("baseline")  # seed -> run
run = baseline_runs[42]  # reference run for the viewer and the figures
print(f"Found checkpoint at: {run.checkpoint}")
print("Baseline seeds:", {seed: r.path.name for seed, r in baseline_runs.items()})

Found checkpoint at: /home/rl_unitree_go2/logs/flat/2026-11-11_18-27-30_baseline/model_499.pt
Baseline seeds: {42: '2026-11-11_18-27-30_baseline', 1: '2026-11-12_10-51-16_baseline', 2: '2026-11-12_11-06-06_baseline'}


## 1.4 Evaluation

### 1.4a Testing your policy 

Watch your trained policy in the viewer. 
Does it follow the commands? 
Does the gait look natural, or does it shuffle, hop or vibrate? 
Is it stable when the command changes? 
What you see here guides your next reward changes!
Stop the viewer (■) when done.

In [20]:
play.interactive(
    env_cfg=make_env_cfg(play=True),
    rl_cfg=make_rl_cfg(),
    policy=run.checkpoint,
)

Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 8      |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----------+------------------------+
| Index | Name     |          

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: https://gnoto.epfl.ch/user-redirect/proxy/8080


(viser) Connection opened (0, 1 total), 262 persistent messages

(viser) Connection opened (1, 2 total), 262 persistent messages

(viser) Connection closed (1, 1 total)


Ctrl+C received. Shutting down viewer...


(viser) Connection closed (0, 0 total)

(viser) Server stopped

Viewer closed!


/home/gnoto_venvs/lr-project/lib/python3.13/site-packages/mjlab/viewer/viser/reward_bar_panel.py:86: UserWarning: Attempted to remove an already removed GuiHtmlHandle.
  self._html_handle.remove()


### 1.4b Collecting rollout data for analysis

`collect_rollouts` runs the policy on a fixed grid of commands, with `robots_per_command` robots per command for `duration_s` seconds, and records everything (base motion, joint states, torques, foot contacts and ground reaction forces). 
The data is saved as a `Rollout` file in the run directory, under `eval/<name>_<checkpoint>.npz`, and is what you analyse in notebook 02.
For more information on what exactly gets recorded, check out `src/lr_project/rollout.py`.

- Running the cell again **loads** the saved file instead of simulating again, unless the settings changed (or `overwrite=True`).
- Use the **same commands and name** for every run you want to compare, so the comparison is fair.

In [22]:
# Fixed commands to evaluate: all combinations of the values below
commands = evaluate.command_grid(
    # Same ranges as the training commands, plus standing still
    vx=(-1.0, 0.0, 1.0), # m/s
    vy=(-0.5, 0.0, 0.5), # m/s
    wz=(-1.0, 0.0, 1.0), # rad/s
)

# Name of the saved evaluation, reuse it for every run you compare
eval_name = "command_grid"

# Same evaluation (commands, robots, duration, evaluation seed) for every training seed
for seed, r in baseline_runs.items():
    evaluate.collect_rollouts(
        env_cfg=make_env_cfg(play=True),
        rl_cfg=make_rl_cfg(),
        checkpoint=r.checkpoint,
        commands=commands,
        name=eval_name,
        robots_per_command=10, # robots per unique command combination
        duration_s=10.0,
        seed=42,
    )
    print(f"seed {seed}: saved evaluation at {r.eval_file(eval_name)}")

[evaluate] loaded /home/rl_unitree_go2/logs/flat/2026-11-11_18-27-30_baseline/eval/command_grid_model_499.npz (already evaluated)
seed 42: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-11_18-27-30_baseline/eval/command_grid_model_499.npz
[evaluate] loaded /home/rl_unitree_go2/logs/flat/2026-11-12_10-51-16_baseline/eval/command_grid_model_499.npz (already evaluated)
seed 1: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-12_10-51-16_baseline/eval/command_grid_model_499.npz
[evaluate] loaded /home/rl_unitree_go2/logs/flat/2026-11-12_11-06-06_baseline/eval/command_grid_model_499.npz (already evaluated)
seed 2: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-12_11-06-06_baseline/eval/command_grid_model_499.npz


## 1.5 Controlled experiment

⚠️ Start this section only **after** having a working baseline you are satisfied with and will **not tune further anymore**!

Now test one design choice of your own that is different from your baseline (e.g., a potential improvement).
A controlled experiment changes **exactly one factor** relative to your baseline and keeps everything else fixed: training budget, evaluation commands and analysis. 
Possible types of experiments:

| Type | What changes | Example |
| --- | --- | --- |
| Ablation | Remove a component | Remove an observation term or a penalty |
| Addition | Add a component | Add a reward term or observation noise |
| Substitution | Replace a design choice | A different action space or command distribution |
| Sensitivity | One parameter at a few levels | Action scale, a reward weight, `gamma` |

Results vary between seeds, so one run per configuration can mislead! 
If time allows, train the baseline and the variant with several seeds each.

### 1.5a Experiment design

**[ANSWER]** Fill this in **before** training your new variant:

- **Question:** what do you want to find out?
- **Type:** ablation, addition, substitution or sensitivity.
- **Change:** exactly what differs from the baseline.
- **Held fixed:** what stays the same.
- **Possible confounders:** what else might change as a side effect (e.g. a new action scale also changes the size of the action-rate penalty), and how you deal with it.
- **Prediction:** which metrics you expect to change, in which direction, and why.

- **Question:** Does the policy need the base linear velocity as an observation to track velocity commands, given that the real Go2 cannot measure it directly (it needs a state estimator)?
- **Type:** Ablation.
- **Change:** `base_lin_vel` is removed from the **actor** observations (48 to 45 inputs). The critic still receives it as privileged information.
- **Held fixed:** All other observations, actions, rewards, terminations, events, PPO hyperparameters, the 500-iteration budget, the seeds (42, 1, 2) and the evaluation (same command grid, `eval_name` and evaluation seed).
- **Possible confounders:** The actor input size changes, which is negligible for a 128x128x128 MLP. Keeping the critic unchanged avoids also changing the value estimate. Seed variance is handled by comparing 3 seeds per configuration.
- **Prediction:** Linear velocity tracking degrades (higher `lin_vel_error`, lower `track_linear_velocity` reward), most at high and lateral commands, because a memoryless MLP must infer its own velocity from joint states and the last action. Yaw tracking and falls should change little, since `base_ang_vel` and gravity (IMU) are still observed.

### 1.5b Building variant config

Start from a copy of your baseline config and change exactly one thing. 
Change **either** the env config or rl config (not both) depending on your question.
Generally, env config changes lead to more interesting questions.

In [23]:
VARIANT_NAME = "no_base_lin_vel"

def make_variant_env_cfg(play: bool = False, seed: int = 42) -> ManagerBasedRlEnvCfg:
    base_cfg = make_env_cfg(play=play, seed=seed)

    variant_cfg = copy.deepcopy(base_cfg)

    # Ablation: the actor no longer observes the base linear velocity.
    # The critic keeps it (privileged information, only used during training).
    variant_cfg.observations["actor"].terms.pop("base_lin_vel")

    return variant_cfg

def make_variant_rl_cfg(seed: int = 42) -> RslRlOnPolicyRunnerCfg:
    base_cfg = make_rl_cfg(seed=seed)

    variant_cfg = copy.deepcopy(base_cfg)

    # Unchanged: this is an env cfg variant

    return variant_cfg

### 1.5c Visualize your variant

Check that the variant builds and behaves as intended before spending GPU time on it:

In [24]:
env_cfg = make_variant_env_cfg(play=True)
play.interactive(env_cfg, policy="random") # "random" or "zero"

Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 8      |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----------+------------------------+
| Index | Name     |          

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: https://gnoto.epfl.ch/user-redirect/proxy/8080


(viser) Connection opened (0, 1 total), 59 persistent messages

(viser) Connection opened (1, 2 total), 262 persistent messages

(viser) Connection opened (2, 3 total), 262 persistent messages

... (2 outputs left out)


Ctrl+C received. Shutting down viewer...


(viser) Connection closed (2, 0 total)

(viser) Server stopped

Viewer closed!


### 1.5d Train your variant

Same experiment as the baseline, with your variant name. 
Then load the variant's latest run.

In [25]:
for seed in SEEDS:
    if seed in runs_per_seed(VARIANT_NAME):
        print(f"seed {seed}: already trained, skipped")
        continue
    train.train(
        env_cfg=make_variant_env_cfg(seed=seed),
        rl_cfg=make_variant_rl_cfg(seed=seed),
        experiment="flat",
        variant=VARIANT_NAME,
    )

[train] run dir: /home/rl_unitree_go2/logs/flat/2026-11-12_11-37-51_no_base_lin_vel
[train] device: cuda:0, envs: 4096, seed: 42
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 4096   |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+--------------------------------------

In [26]:
variant_runs = runs_per_seed(VARIANT_NAME)  # seed -> run
variant_run = variant_runs[42]
print(f"Found checkpoint at: {variant_run.checkpoint}")
print("Variant seeds:", {seed: r.path.name for seed, r in variant_runs.items()})

Found checkpoint at: /home/rl_unitree_go2/logs/flat/2026-11-12_11-37-51_no_base_lin_vel/model_499.pt
Variant seeds: {42: '2026-11-12_11-37-51_no_base_lin_vel', 1: '2026-11-12_11-50-50_no_base_lin_vel', 2: '2026-11-12_12-00-11_no_base_lin_vel'}


### 1.5e Test your variant policy

Compare qualitatively with what you saw for the baseline in 1.4a. 
The quantitative comparison will take place in notebook 02.

In [27]:
play.interactive(
    env_cfg=make_variant_env_cfg(play=True),
    rl_cfg=make_variant_rl_cfg(),
    policy=variant_run.checkpoint,
)

Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 8      |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----------+------------------------+
| Index | Name     |          

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

URL to interactive simulation: https://gnoto.epfl.ch/user-redirect/proxy/8080


(viser) Connection opened (0, 1 total), 262 persistent messages

(viser) Connection closed (0, 0 total)

(viser) Connection opened (1, 1 total), 262 persistent messages


Ctrl+C received. Shutting down viewer...


(viser) Connection closed (1, 0 total)

(viser) Server stopped

Viewer closed!


### 1.5f Collect variant rollouts

Use exactly the same `commands` and `eval_name` as for the baseline (1.4b), so that notebook 02 compares them equally.

In [28]:
for seed, r in variant_runs.items():
    evaluate.collect_rollouts(
        env_cfg=make_variant_env_cfg(play=True),
        rl_cfg=make_variant_rl_cfg(),
        checkpoint=r.checkpoint,
        commands=commands,
        name=eval_name,
        robots_per_command=10,
        duration_s=10.0,
        seed=42,
    )
    print(f"seed {seed}: saved evaluation at {r.eval_file(eval_name)}")

[evaluate] creating 27 commands x 10 robots = 270 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 270    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> contains 1 active terms.
+-------------------------------------------+
|            Active Command Terms           |
+-------+----

100%|██████████| 500/500 [00:07<00:00, 69.95it/s]


seed 42: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-12_11-37-51_no_base_lin_vel/eval/command_grid_model_499.npz
[evaluate] creating 27 commands x 10 robots = 270 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 270    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> co

100%|██████████| 500/500 [00:07<00:00, 68.76it/s]


seed 1: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-12_11-50-50_no_base_lin_vel/eval/command_grid_model_499.npz
[evaluate] creating 27 commands x 10 robots = 270 environments
Setting seed: 42

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 270    |
| Environment device     | cuda:0 |
| Environment seed       | 42     |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 1 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+-----------+-------------------------+
|   Index   | Name                    |
+-----------+-------------------------+
|     0     | reset_base              |
|     1     | reset_joints            |
+-----------+-------------------------+

[INFO] <CommandManager> con

100%|██████████| 500/500 [00:06<00:00, 75.06it/s]


seed 2: saved evaluation at /home/rl_unitree_go2/logs/flat/2026-11-12_12-00-11_no_base_lin_vel/eval/command_grid_model_499.npz


## Next step

You now have a baseline and a variant, each with training logs and evaluation rollouts.
Continue in `02_flat_analysis.ipynb`.